# SmartRenew Analytics — Machine LearningCe notebook contient les deux volets de Machine Learning du projet :1. **Clustering (K-Means, k=6)** — regroupe les ~35 800 communes en profils types, séparément pour le solaire et l'éolien.2. **Régression temporelle** — projette la production de chaque commune aux horizons 2030, 2035 et 2040.Les résultats sont écrits dans la table `mart_ml_communes`, ensuite jointe au dashboard via la vue `mart_dashboard_ml`.**Stack :** Python · scikit-learn · pandas · BigQuery

## 1. Configuration & chargement des données> **Authentification :** ce notebook lit la variable d'environnement `GOOGLE_APPLICATION_CREDENTIALS`.> Avant de lancer le notebook, définissez-la dans votre terminal (la clé n'est jamais incluse dans le dépôt) :> ```bash> export GOOGLE_APPLICATION_CREDENTIALS="/chemin/vers/votre-cle.json"> ```

In [ ]:
import pandas as pdimport numpy as npfrom google.cloud import bigqueryfrom sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.linear_model import LinearRegressionfrom sklearn.metrics import r2_scoreimport os# L'authentification utilise la variable d'environnement GOOGLE_APPLICATION_CREDENTIALS# (à définir avant de lancer le notebook — voir la cellule Markdown ci-dessus).if not os.environ.get('GOOGLE_APPLICATION_CREDENTIALS'):    raise EnvironmentError(        "Variable GOOGLE_APPLICATION_CREDENTIALS non définie. "        "Lancez : export GOOGLE_APPLICATION_CREDENTIALS='/chemin/vers/cle.json'"    )PROJECT = 'project-final-wagon'DATASET = 'dbt_fortizgonthier_marts'client = bigquery.Client(project=PROJECT)

In [ ]:
# Chargement de la table source (une ligne par commune)df = client.query(f"""    SELECT *    FROM `{PROJECT}.{DATASET}.mart_dashboard`""").to_dataframe()print(f"{len(df)} communes chargées, {df.shape[1]} colonnes")df.head()

## 2. Préparation des featuresLe raccordement réseau est une variable textuelle (« Très favorable », « Favorable »…). On la convertit en score numérique pour pouvoir l'utiliser dans le clustering.

In [ ]:
mapping_raccordement = {    'Très favorable': 3,    'Favorable': 2,    'Modéré': 1,    'Inconnu': 1,   # valeur neutre pour les communes sans info}df['raccordement_num'] = df['score_raccordement'].map(mapping_raccordement).fillna(1)df['score_raccordement'].value_counts()

## 3. Clustering — Profils types de communesOn entraîne **deux modèles K-Means séparés** (k=6), un pour le solaire et un pour l'éolien, car les critères pertinents diffèrent selon la technologie.Chaque jeu de features est standardisé (`StandardScaler`) pour que toutes les variables pèsent équitablement dans le calcul des distances.

### 3.1 Clustering solaire

In [ ]:
features_solaire = [    'score_solaire',    'irradiation_kwh_m2_an',    'production_kwh_kwc_an',    'surface_solaire_ha',    'roi_sol_ans',    'prix_terrain_median_eur_ha',    'pct_territoire_protege',    'pente_moy_deg',    'raccordement_num',]X_sol = df[features_solaire].copy()X_sol['roi_sol_ans'] = X_sol['roi_sol_ans'].clip(0, 50)X_sol = X_sol.fillna(X_sol.median(numeric_only=True))scaler_sol = StandardScaler()X_sol_scaled = scaler_sol.fit_transform(X_sol)km_sol = KMeans(n_clusters=6, random_state=42, n_init=10)df['cluster_solaire'] = km_sol.fit_predict(X_sol_scaled)print(df['cluster_solaire'].value_counts().sort_index())

In [ ]:
profil_sol = df.groupby('cluster_solaire')[features_solaire].mean().round(1)profil_sol['nb_communes'] = df['cluster_solaire'].value_counts().sort_index()profil_sol

In [ ]:
# Nommage des profils solaires (à ajuster selon l'inspection ci-dessus)noms_solaire = {    0: "🏞️ Vaste foncier — projets d'envergure",    1: "✅ Accès optimal — meilleur compromis",    2: "🌾 Plaine Nord — potentiel modéré",    3: "⭐ Premium Sud — fort potentiel (vigilance zones)",    4: "🏙️ Sans foncier disponible",    5: "🛡️ Zone protégée — faible potentiel",}df['profil_solaire'] = df['cluster_solaire'].map(noms_solaire)df['profil_solaire'].value_counts()

### 3.2 Clustering éolien

In [ ]:
features_eolien = [    'score_eolien',    'wind_speed_moy_ms',    'productible_eolien_mwh_an',    'surface_eolien_ha',    'roi_eol_ans',    'prix_terrain_median_eur_ha',    'pct_territoire_protege',    'pente_moy_deg',    'altitude_moy_m',    'raccordement_num',]X_eol = df[features_eolien].copy()X_eol['roi_eol_ans'] = X_eol['roi_eol_ans'].clip(0, 50)X_eol = X_eol.fillna(X_eol.median(numeric_only=True))scaler_eol = StandardScaler()X_eol_scaled = scaler_eol.fit_transform(X_eol)km_eol = KMeans(n_clusters=6, random_state=42, n_init=10)df['cluster_eolien'] = km_eol.fit_predict(X_eol_scaled)print(df['cluster_eolien'].value_counts().sort_index())

In [ ]:
profil_eol = df.groupby('cluster_eolien')[features_eolien].mean().round(1)profil_eol['nb_communes'] = df['cluster_eolien'].value_counts().sort_index()profil_eol

In [ ]:
# Nommage des profils éoliens (à ajuster selon l'inspection)noms_eolien = {    0: "🌬️ Vent modéré — accessible",    1: "⭐ Excellent vent (sous contrainte zones)",    2: "🛡️ Faible vent — protégé",    3: "⛰️ Relief difficile — non viable",    4: "⭐ Excellent — vent fort, accès libre",    5: "🌾 Plaine dégagée — favorable",}df['profil_eolien'] = df['cluster_eolien'].map(noms_eolien)df['profil_eolien'].value_counts()

## 4. Régression temporelle — Projections 2030 / 2035 / 2040Pour chaque commune, on ajuste une **régression linéaire** sur la production totale historique (2017-2024), puis on projette aux horizons 2030, 2035 et 2040. C'est une **projection tendancielle** : elle prolonge la dynamique passée.

In [ ]:
annees = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]cols_prod = [f'prod_tot_{a}_mwh' for a in annees]X_temps = np.array(annees).reshape(-1, 1)horizons = [2030, 2035, 2040]preds = {h: [] for h in horizons}tendances, r2s = [], []for _, row in df.iterrows():    y = row[cols_prod].values.astype(float)    y = np.nan_to_num(y, nan=0.0)    reg = LinearRegression().fit(X_temps, y)    for h in horizons:        preds[h].append(max(0, reg.predict([[h]])[0]))    tendances.append(reg.coef_[0])    y_pred = reg.predict(X_temps)    r2s.append(r2_score(y, y_pred) if np.std(y) > 0 else 0.0)df['pred_prod_2030_mwh'] = np.round(preds[2030], 1)df['pred_prod_2035_mwh'] = np.round(preds[2035], 1)df['pred_prod_2040_mwh'] = np.round(preds[2040], 1)df['tendance_prod_mwh_an'] = np.round(tendances, 2)df['fiabilite_tendance_r2'] = np.round(r2s, 3)for h in horizons:    print(f"Projection {h} : {df[f'pred_prod_{h}_mwh'].sum() / 1e6:.1f} TWh")

### Fiabilité du modèle : deux niveaux de lectureLa régression temporelle est volontairement simple (projection tendancielle à partir de la seule variable temporelle). Sa fiabilité se lit à deux échelles :- **Échelle nationale (R² ≈ 0.98)** : la production ENR agrégée suit une tendance de croissance très régulière. C'est ce niveau qui fonde nos projections nationales, et il est très fiable.- **Échelle communale (R² moyen ≈ 0.56)** : commune par commune, la production évolue par paliers d'installation (l'ajout d'une éolienne crée un saut brutal), ce qui limite mécaniquement le R² d'un modèle linéaire.Ces variations locales se compensent à l'échelle agrégée : c'est pourquoi les projections nationales restent robustes malgré un R² communal modéré.*Note méthodologique : une régression multivariée (prédire la production à partir des caractéristiques de la commune) a été écartée — testée, elle donne un R² négatif et serait en partie tautologique, certaines features étant dérivées de la production elle-même. La régression temporelle est l'approche la plus robuste et la plus honnête pour ce problème.*

In [ ]:
# Fiabilité à deux échellesprod_nat = [df[f'prod_tot_{a}_mwh'].sum()/1e6 for a in annees]reg_nat = LinearRegression().fit(X_temps, prod_nat)r2_national = r2_score(prod_nat, reg_nat.predict(X_temps))actives = df[df['prod_tot_2024_mwh'] > 0]r2_communal = actives['fiabilite_tendance_r2'].mean()print(f"R² national (projections agrégées) : {r2_national:.3f}")print(f"R² communal moyen (communes actives) : {r2_communal:.3f}")

## 5. Écriture des résultats dans BigQueryOn ne conserve que les colonnes ML (plus la clé `code_insee`) et on écrit la table `mart_ml_communes`.> **Note d'architecture :** `mart_ml_communes` est produite par ce notebook Python (pas par dbt). dbt la consomme ensuite en lecture via la vue `mart_dashboard_ml`, qui joint le dashboard et les résultats ML.

In [ ]:
cols_ml = [    'code_insee', 'nom_commune',    'cluster_solaire', 'cluster_eolien',    'profil_solaire', 'profil_eolien',    'pred_prod_2030_mwh', 'pred_prod_2035_mwh', 'pred_prod_2040_mwh',    'tendance_prod_mwh_an', 'fiabilite_tendance_r2',]df_ml = df[cols_ml].copy()table_id = f'{PROJECT}.{DATASET}.mart_ml_communes'job_config = bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')client.load_table_from_dataframe(df_ml, table_id, job_config=job_config).result()print(f"✅ {len(df_ml)} communes écrites dans {table_id}")

In [ ]:
# Recréer la vue mart_dashboard_ml (dashboard + ML)client.query(f"""CREATE OR REPLACE VIEW `{PROJECT}.{DATASET}.mart_dashboard_ml` ASSELECT d.*,    ml.cluster_solaire, ml.cluster_eolien, ml.profil_solaire, ml.profil_eolien,    ml.pred_prod_2030_mwh, ml.pred_prod_2035_mwh, ml.pred_prod_2040_mwh,    ml.tendance_prod_mwh_an, ml.fiabilite_tendance_r2FROM `{PROJECT}.{DATASET}.mart_dashboard` dLEFT JOIN `{PROJECT}.{DATASET}.mart_ml_communes` ml    ON d.code_insee = ml.code_insee""").result()print("✅ Vue mart_dashboard_ml recréée")